# Qwen4B original chunks — full, part 2/4
Scores exact submitted chunks; no submission is built on Kaggle.

In [ ]:
%pip install -q --no-cache-dir 'transformers>=4.51,<5' 'accelerate>=1.6,<2' 'huggingface-hub>=0.30,<1' 'pyarrow>=18,<22' 'pandas>=2.2,<3'

In [ ]:
import hashlib,json,os,subprocess,sys,time,zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import torch
SCOPE='full'; PART_INDEX=1; PART_COUNT=4
MODEL_ID='Qwen/Qwen3-Reranker-4B'; REVISION='22e683669bc0f0bd69640a1354a6d0aebcfeede5'; INSTRUCTION='Given a multilingual medical question, retrieve passages that contain information needed to answer the question while respecting clinical intent, population, treatment, and negation constraints.'
WORK=Path('/kaggle/working/qwen_chunks'); WORK.mkdir(parents=True,exist_ok=True)
try:
 from kaggle_secrets import UserSecretsClient
 token=UserSecretsClient().get_secret('HF_TOKEN')
 if token: os.environ['HF_TOKEN']=token
except Exception: pass
assert torch.cuda.device_count()==2,torch.cuda.device_count()
root=Path('/kaggle/input'); pairs_found=list(root.rglob('qwen_chunk_pairs.parquet')); manifests=list(root.rglob('qwen_chunk_manifest.json'))
if len(pairs_found)!=1 or len(manifests)!=1: raise RuntimeError({'pairs':pairs_found,'manifests':manifests})
manifest=json.loads(manifests[0].read_text(encoding='utf-8')); assert manifest['scope']==SCOPE
pairs=pd.read_parquet(pairs_found[0]); qids=sorted(pairs.query_id.astype(int).unique())
owned={qid for i,qid in enumerate(qids) if i%PART_COUNT==PART_INDEX}
pairs=pairs[pairs.query_id.isin(owned)].copy().reset_index(drop=True)
input_path=WORK/'pairs.parquet'; pairs.to_parquet(input_path,index=False,compression='zstd')
print({'scope':SCOPE,'part':f'{PART_INDEX+1}/{PART_COUNT}','queries':len(owned),'pairs':len(pairs),'gpus':[torch.cuda.get_device_name(i) for i in range(2)]})

In [ ]:
WORKER='"""GPU worker for exact, non-truncated Qwen4B chunk scoring."""\nfrom __future__ import annotations\n\nimport argparse\nimport gc\nimport json\nimport time\nfrom pathlib import Path\n\nimport pandas as pd\nimport torch\nfrom transformers import AutoModelForCausalLM, AutoTokenizer\n\n\nPREFIX = \'<|im_start|>system\\nJudge whether the Document meets the requirements based on the Query and the Instruct provided. Note that the answer can only be "yes" or "no".<|im_end|>\\n<|im_start|>user\\n\'\nSUFFIX = \'<|im_end|>\\n<|im_start|>assistant\\n<think>\\n\\n</think>\\n\\n\'\n\n\ndef main() -> None:\n    p = argparse.ArgumentParser()\n    p.add_argument("--input", type=Path, required=True); p.add_argument("--out", type=Path, required=True)\n    p.add_argument("--model-id", required=True); p.add_argument("--revision", required=True)\n    p.add_argument("--gpu", type=int, required=True); p.add_argument("--world-size", type=int, default=2)\n    p.add_argument("--max-length", type=int, default=4096); p.add_argument("--instruction", required=True)\n    a = p.parse_args()\n    torch.cuda.set_device(0); device = torch.device("cuda:0")\n    frame = pd.read_parquet(a.input).reset_index(drop=True)\n    qids = sorted(frame.query_id.astype(int).unique())\n    owned = {qid for i, qid in enumerate(qids) if i % a.world_size == a.gpu}\n    frame = frame[frame.query_id.isin(owned)].copy().reset_index(drop=True)\n    a.out.mkdir(parents=True, exist_ok=True)\n    checkpoint = a.out / f"qwen4b_gpu{a.gpu}.parquet"\n    done = set()\n    old = None\n    if checkpoint.exists():\n        old = pd.read_parquet(checkpoint); done = set(old.pair_id.astype(str))\n    pending = frame[~frame.pair_id.astype(str).isin(done)].copy().reset_index(drop=True)\n    print(json.dumps({"event":"start","gpu":a.gpu,"rows":len(frame),"pending":len(pending)}), flush=True)\n\n    tok = AutoTokenizer.from_pretrained(a.model_id, revision=a.revision, padding_side="left")\n    model = AutoModelForCausalLM.from_pretrained(\n        a.model_id, revision=a.revision, torch_dtype=torch.float16, attn_implementation="sdpa"\n    ).to(device).eval()\n    if tok.pad_token_id is None: tok.pad_token = tok.eos_token\n    yes_id = tok.convert_tokens_to_ids("yes"); no_id = tok.convert_tokens_to_ids("no")\n    prefix = tok.encode(PREFIX, add_special_tokens=False); suffix = tok.encode(SUFFIX, add_special_tokens=False)\n    revision = getattr(model.config, "_commit_hash", None) or a.revision\n\n    # Qwen3ForCausalLM otherwise projects every sequence position to the full\n    # 151k-token vocabulary even though reranking only consumes the last token.\n    # Verify once that last-token-only logits are numerically identical before\n    # using the cheaper path for the full shard.\n    if len(pending):\n        probe = pending.sort_values("passage_chars").iloc[0]\n        probe_text = f"<Instruct>: {a.instruction}\\n<Query>: {probe.query_text}\\n<Document>: {probe.passage_text}"\n        probe_ids = prefix + tok.encode(probe_text, add_special_tokens=False) + suffix\n        if len(probe_ids) > a.max_length:\n            raise RuntimeError("Shortest optimization probe exceeds max_length")\n        probe_tensor = torch.tensor([probe_ids], dtype=torch.long, device=device)\n        probe_mask = torch.ones_like(probe_tensor)\n        with torch.inference_mode():\n            full_last = model(input_ids=probe_tensor, attention_mask=probe_mask).logits[:, -1, [no_id, yes_id]].float()\n            kept_last = model(\n                input_ids=probe_tensor, attention_mask=probe_mask, logits_to_keep=1\n            ).logits[:, -1, [no_id, yes_id]].float()\n        probe_diff = float((full_last - kept_last).abs().max().item())\n        if probe_diff > 1e-5:\n            raise RuntimeError(f"logits_to_keep self-check failed: {probe_diff}")\n        print(json.dumps({"event":"last_token_logits_self_check","max_abs_diff":probe_diff,\n                          "probe_tokens":len(probe_ids)}), flush=True)\n        del full_last, kept_last, probe_tensor, probe_mask\n        torch.cuda.empty_cache()\n\n    rows = [] if old is None else old.to_dict("records")\n    started = time.time(); since_save = 0\n    with torch.inference_mode():\n        for index, row in enumerate(pending.itertuples(index=False), 1):\n            formatted = f"<Instruct>: {a.instruction}\\n<Query>: {row.query_text}\\n<Document>: {row.passage_text}"\n            body = tok.encode(formatted, add_special_tokens=False)\n            ids = prefix + body + suffix\n            if len(ids) > a.max_length:\n                raw = norm = None; status = "too_long"\n            else:\n                tensor = torch.tensor([ids], dtype=torch.long, device=device)\n                logits = model(\n                    input_ids=tensor, attention_mask=torch.ones_like(tensor), logits_to_keep=1\n                ).logits[:, -1, :].float()\n                raw = float((logits[:, yes_id] - logits[:, no_id]).item())\n                norm = float(torch.sigmoid(torch.tensor(raw)).item()); status = "ok"\n            rows.append({\n                "pair_id": str(row.pair_id), "query_id": int(row.query_id),\n                "chunk_ordinal": int(row.chunk_ordinal), "doc_id": int(row.doc_id),\n                "raw_score": raw, "normalized_score": norm, "input_tokens": len(ids),\n                "status": status, "model_revision": revision, "gpu_shard": a.gpu,\n                "query_sha256": str(row.query_sha256), "passage_sha256": str(row.passage_sha256),\n            })\n            since_save += 1\n            if since_save >= 128 or index == len(pending):\n                pd.DataFrame(rows).to_parquet(checkpoint, index=False, compression="zstd"); since_save = 0\n            if index % 100 == 0 or index == len(pending):\n                elapsed = time.time() - started\n                print(json.dumps({"event":"progress","gpu":a.gpu,"complete":index,"total":len(pending),\n                                  "pairs_s":round(index/max(elapsed,1e-6),3),"status":status}), flush=True)\n    result = pd.read_parquet(checkpoint)\n    if set(result.pair_id.astype(str)) != set(frame.pair_id.astype(str)):\n        raise RuntimeError("Checkpoint coverage mismatch")\n    del model; gc.collect(); torch.cuda.empty_cache()\n    print(json.dumps({"event":"complete","gpu":a.gpu,"rows":len(result),\n                      "statuses":result.status.value_counts().to_dict()}), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'
worker_path=WORK/'worker.py'; worker_path.write_text(WORKER,encoding='utf-8'); print({'worker_sha256':hashlib.sha256(WORKER.encode()).hexdigest()})

In [ ]:
processes=[]; logs=[]
for gpu in range(2):
 out=WORK/'checkpoints'/f'gpu{gpu}'; out.mkdir(parents=True,exist_ok=True)
 path=WORK/f'gpu{gpu}.log'; log=path.open('w',encoding='utf-8'); logs.append((log,path))
 cmd=[sys.executable,str(worker_path),'--input',str(input_path),'--out',str(out),'--model-id',MODEL_ID,'--revision',REVISION,'--gpu',str(gpu),'--max-length','4096','--instruction',INSTRUCTION]
 env=os.environ.copy(); env['CUDA_VISIBLE_DEVICES']=str(gpu); env['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'
 processes.append(subprocess.Popen(cmd,stdout=log,stderr=subprocess.STDOUT,env=env))
while any(p.poll() is None for p in processes):
 time.sleep(60); state={'returncodes':[p.poll() for p in processes]}
 for _,path in logs:
  if path.exists(): state[path.name]=path.read_text(encoding='utf-8',errors='replace').splitlines()[-1:]
 print(state,flush=True)
for log,_ in logs: log.close()
for process,(_,path) in zip(processes,logs):
 if process.returncode!=0: raise RuntimeError(path.read_text(encoding='utf-8',errors='replace')[-8000:])
frames=[pd.read_parquet(WORK/'checkpoints'/f'gpu{gpu}'/f'qwen4b_gpu{gpu}.parquet') for gpu in range(2)]
scores=pd.concat(frames,ignore_index=True).sort_values(['query_id','chunk_ordinal']).reset_index(drop=True)
assert len(scores)==len(pairs) and scores.pair_id.nunique()==len(pairs)
scores.to_parquet(WORK/'qwen4b_chunk_scores.parquet',index=False,compression='zstd')
validation={'schema_version':'qwen4b_original_chunk_scores_v1','scope':SCOPE,'part_index':PART_INDEX,'part_count':PART_COUNT,'queries':int(scores.query_id.nunique()),'pairs':len(scores),'statuses':scores.status.value_counts().to_dict(),'control_sha256':manifest['control_zip_sha256'],'input_sha256':manifest['artifacts']['qwen_chunk_pairs.parquet']['sha256'],'model_id':MODEL_ID,'revision':REVISION,'max_length':4096}
(WORK/'validation.json').write_text(json.dumps(validation,indent=2),encoding='utf-8')
(WORK/'run_manifest.json').write_text(json.dumps({'input':manifest,'validation':validation,'instruction':INSTRUCTION},ensure_ascii=False,indent=2),encoding='utf-8')
name=f'qwen4b_chunks_{SCOPE}_part{PART_INDEX+1:02d}of{PART_COUNT:02d}.zip'; target=Path('/kaggle/working')/name
with zipfile.ZipFile(target,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=9) as z:
 for name in ['qwen4b_chunk_scores.parquet','validation.json','run_manifest.json']: z.write(WORK/name,arcname=name)
assert zipfile.ZipFile(target).testzip() is None
print(validation); print({'output':str(target),'bytes':target.stat().st_size})